# Step 1: Importing the necessary libraries 
The first thing to do is to properly set up the notebook by importing the necessary libraires. 

In [ ]:
#Import the necessary libraires 
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm

# Step 2: Loading the datasets
The bathymetry and chlorophyll-a datasets that will be used for this assignment need to be loaded

In [ ]:
#---------------------------------------------------------BATHYMETRY DATA------------------------------------------------------------
#Loading and inspecting the Bathymetry data 
bath = xr.open_dataset('data/GMRTv4_4_1_20260513topo.grd', engine='netcdf4')
bath.head() #see what names are used for the attributes (columns)

#Get the max and min altitude/elevation values to be used for later
print("Bath Min", bath.altitude.min().values)
print("Bath Max", bath.altitude.max().values)

In [ ]:
#----------------------------------------------------CHLOROPHYLL-A DATA--------------------------------------------------------------
#Loading and inspecting the Chlorophyll-a data
chlor = xr.open_dataset('data/ESACCI-OC-MAPPED-CLIMATOLOGY-1M_MONTHLY_4km_PML_CHL-fv5.0.nc')
chlor.head() #see what names are used for the attributes (columns) 

#Get the max and min values of chlor_a to be used for later 
print("Chlor Min", chlor.chlor_a.min().values)
print("Chlor Max", chlor.chlor_a.max().values)

# Step 3: Make a Bathymetry Map 
Using the downloaded Bathymetry data for the extent encapsulating the Galapagos Archipelago

In [ ]:
#--------------------------------------------------------------Plot figure-----------------------------------------------------------
plt.figure(figsize=(12,7))
ax = plt.axes(projection=ccrs.PlateCarree())


#------------------------------------------------------Seperating the data-----------------------------------------------------------
#The altitude values range from + to -.
# + values = terrain elevation 
# - values = depth of the sea floor 
#Seperate the data 
sea_depth = bath.altitude.where(bath.altitude <= 0)
land_height = bath.altitude.where(bath.altitude > 0)

#Plotting the sea_depth data
sea_data = sea_depth.plot(ax=ax, 
                          cmap="Blues_r", #suitable to display the ocean) 
                          vmin=-5000, #max depth for the extent 
                          vmax=0, 
                          add_colorbar=False #will add a custom colourbar manually 
                         )

#Plotting the land_height data 
land_data = land_height.plot(ax=ax, 
                             cmap='YlOrBr', #suitable for displaying terrain 
                             vmin=0, 
                             vmax=3000, 
                             add_colorbar=False #will add a custom colourbar manually
                            )


#------------------------------------------------------Manually adding colorbars------------------------------------------------------
#Manually adding the colorbars for sea and land 
cbar_sea = plt.colorbar(sea_data, ax=ax, orientation='vertical', pad=0.02)
cbar_sea.set_label('Sea Depth (m)', fontsize=12, fontweight='bold', labelpad=15)
cbar_land = plt.colorbar(land_data, ax=ax, orientation='vertical', pad=0.08)
cbar_land.set_label('Land Elevation (m)', fontsize=12, fontweight='bold', labelpad=15)


#----------------------------------------------------------North Arrow---------------------------------------------------------------
#I want to comply with the cartographic principals
#creating a function to add in the North Arrow 
def add_north_arrow(ax, location=(0.92, 0.92), size=10):
    """
    location: (x, y) in axes coordinates (0 to 1)
    size: size of the 'N' text
    """
    x, y = location
    #Need to 'draw' the arrow 
    ax.annotate('↑', xy=(x, y), xytext=(x, y-0.02),
                xycoords='axes fraction',
                ha='center', va='center',
                fontsize=size + 10, fontweight='bold')
    #Add in the 'N' 
    ax.annotate('N', xy=(x, y), xytext=(x, y+0.03),
                xycoords='axes fraction',
                ha='center', va='center',
                fontsize=size, fontweight='bold')

#Adding the north arrow to the plot
add_north_arrow(ax)

#I did try to add a scalebar but there were far too many errors and I could not figure it out!
#Even AI could not assist

#---------------------------------------------------------Island Labels--------------------------------------------------------------
#Got the coordinates for the islands and defined them with their names
#Moved the coordinates to move labels off the central peaks
islands = {
    'Isabela': (-91.1, -0.9),      #shifted south/west
    'Fernandina': (-91.8, -0.45),  #shifted west into the water
    'Santa Cruz': (-90.35, -0.6),   #shifted east
    'Santiago': (-90.7, -0.1)      #shifted north
}

#used a for loop to apply the lables to the islands
for name, (lon, lat) in islands.items():
    ax.text(lon, lat, name, 
            transform=ccrs.PlateCarree(),
            fontsize=6, 
            fontweight='bold',
            fontstyle='italic',
            ha='center', 
            va='center'
           )

#---------------------------------------------------------Final touches--------------------------------------------------------------
#Formatting the plot 
ax.coastlines()
gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
gl.top_labels=False 
gl.right_labels=False
plt.title("Bathymetry of The Galapagos Archipelago", fontsize=16, fontweight='bold', pad=20)


#---------------------------------------------------------Save the figure------------------------------------------------------------
#Save the figure for the pdf document 
plt.savefig('Galap_Bath_Map.png', dpi=300, bbox_inches='tight')


#-------------------------------------------------Displaying the figure--------------------------------------------------------------
plt.show()

# Step 4: Creating a plot of the Mean Annual Chlorophyll-a for the Galapagos Archipelago

The Chlorophyll-a data provided was a global dataset, so the first thing I did was to slice the data so that its extent only encapsulated The Galapagos Archipelago

In [ ]:
#--------------------------------------------------Create the Galapagos variable---------------------------------------------------------------
#Creating a variable that contains the Chlorophyll-a data for the Galapagos Archipelago Extent 
galap_ch = chlor.sel(lat=slice(1.5, -1.5), lon=slice(-92.5, -90.0))

#Saving the slice to a new NetCDF file, dont need the whole file, as its too big 
galap_ch.to_netcdf('galapagos_chlorophyll_slice.nc')

#Inspect the variable
galap_ch.head()

#Max and min needed for the normLog function
print("Min", galap_ch.chlor_a.min().values)
print("Max", galap_ch.chlor_a.max().values)

In [ ]:
#--------------------------------------------------Plotting the Data--------------------------------------------------------------
#Plotting the Chlorophyll-a data 
plt.figure(figsize=(10,6))
ax = plt.axes(projection=ccrs.PlateCarree())
mean_chlor = galap_ch.chlor_a.mean(dim='time')
im = mean_chlor.plot(ax=ax, norm=LogNorm(vmin=0.013, vmax=13.26), cmap='gist_stern')
im.colorbar.set_label('Chlorophyll-a (mg/m$^{-3}$)', fontsize=12, fontweight='bold')
ax.add_feature(cfeature.LAND, facecolor='snow', zorder=2) 
ax.coastlines(resolution='10m', color='black', zorder=3)
plt.title("Annual Mean Chlorophyll-a (mg/m$^{-3}$)", fontsize=16, fontweight='bold', pad=20)


#--------------------------------------------------Save the figure---------------------------------------------------------------
#Save the figure for the pdf document 
plt.savefig('Annual_Mean_Chlor_Galap.png', dpi=300, bbox_inches='tight')


#--------------------------------------------------Display the figure---------------------------------------------------------------
plt.show()

# Step 5: Faceted Monthly plots of the Chlorophyll-a concentration of the Galapagos Archipelago 

In [ ]:
#--------------------------------------------------Inspect the data---------------------------------------------------------------
#Inspecting the data to see how the layout of dates 
print(galap_ch.time.values)
#This shows that the time column is not valued correctly
#Some of the 1997 months are scattered between the 1998 months

#Reordering the data using the sortby function 
galap_ch = galap_ch.sortby('time')

#Groupling the data by month 
monthly_ch = galap_ch.chlor_a.groupby('time.month').mean()

In [ ]:
#--------------------------------------------------Plotting the data---------------------------------------------------------------
#Plotting the faceted monthly plots 
fg = monthly_ch.plot(col="month", 
                        col_wrap=4, #4 plots per row 
                        norm=LogNorm(vmin=0.013, vmax=13.26), 
                        cmap='gist_stern', 
                        subplot_kws={'projection' : ccrs.PlateCarree()}
                       )

fg.fig.suptitle("Monthly Chlorophyll-a concentrations of the Galapagos Archipelago", 
               fontsize=20, 
               fontweight='bold', 
               y=1.05)

fg.cbar.set_label('Chlorophyll-a concentration (mg/m$^{-3}$)', fontsize=12, fontweight='bold')

month_names = ['September 1997', 'October 1997', 'November 1997', 
               'December 1997', 'January 1998', 'February 1998', 
               'March 1998', 'April 1998', 'May 1998', 
               'June 1998', 'July 1998', 'August 1998']

for ax, name in zip(fg.axes.flat, month_names):
    ax.add_feature(cfeature.LAND, facecolor='snow', zorder=2)
    ax.coastlines(resolution='10m', color='black', zorder=3)
    ax.set_title(name)


#--------------------------------------------------Save the figure---------------------------------------------------------------
#Save the figure for the pdf document 
plt.savefig('Faceted_Mean_Chlor_Galap.png', dpi=300, bbox_inches='tight')


#--------------------------------------------------Display the figure---------------------------------------------------------------
plt.show()

# Step 6: Timeseries of the Area Mean vs Single Point 

In [ ]:
#--------------------------------------------------Create the Single Point---------------------------------------------------------------
#Need to determine the single point, an upwelling hotspot  
#Western side of the Isabela Island 
point_lat, point_lon = -0.5, -91.5

#Selecting the point from galap_ch.chlor_a as it contians the full 12 month series 
point_data = galap_ch.chlor_a.sel(lat=point_lat, lon=point_lon, method='nearest')
point_data = point_data.sortby('time') #make sure that the data is ordered correctly 

In [ ]:
#--------------------------------------------------Determine the Area Mean---------------------------------------------------------------
#Calculating the mean seasonal cycle for the entire 300km extent around Galapagos Archipelago 
area_mean = galap_ch.chlor_a.mean(dim=['lat', 'lon'])
area_mean = area_mean.sortby('time')

In [ ]:
#--------------------------------------------------Plotting the figure---------------------------------------------------------------
plt.figure(figsize=(10,5))

#plotting the lines 
area_mean.plot(label='Region Mean', linewidth=2, marker='o')
point_data.plot(label='Upwelling Hotspot (Western side of Isabela)', linestyle='--', marker='s')

#Plotting the rest of the elements 
plt.legend()
plt.grid(True, alpha=0.3)

#I was having problem plotting the x and y labels, the metadata was automatically being used as the lables
#even after I used the plt.xlabel() and plt.ylabel()
#I used the following to overwrite the xarray default 
plt.gca().set_ylabel("Chlorophyll-a (mg/m$^{-3}$)", fontsize=12, fontweight='bold')
plt.gca().set_xlabel("Time", fontsize=12, fontweight='bold')

#I did not like the place the legend was automatically being plotted (top right corner)
#It was covering some of the data 
#Used the following to manually place it 
plt.legend(loc='center left', bbox_to_anchor=(0.52, 0.45), frameon=True)

#plotting the title 
plt.title("Seasonal Chlorophyll-a Comparison: Regional Average vs Local Hotspot", fontsize=16, fontweight='bold', pad=15)


#--------------------------------------------------Save the figure---------------------------------------------------------------
#save figure for the pdf 
plt.savefig('Chlorophyll_Comparison_Galap.png', dpi=300, bbox_inches='tight')

#--------------------------------------------------Display the figure--------------------------------------------------------------- 
plt.show()